# 📊 Analyse exploratoire (EDA) — ventes NordRetail

**Objectif** : charger le modèle en étoile NordRetail, l'enrichir par jointures, puis explorer
les données pour en tirer 3-4 premiers enseignements métier.

> **Analogie** — l'EDA, c'est la **visite des lieux avant les travaux** : on regarde ce qu'on a,
> ce qui manque, ce qui cloche, avant de construire quoi que ce soit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = "../../99-Brief/Data-Analyst/data"
pd.set_option("display.max_columns", None)

## 1. Charger les tables du modèle en étoile

In [ ]:
faits   = pd.read_csv(f"{DATA}/Faits_Ventes.csv")
produit = pd.read_csv(f"{DATA}/Dim_Produit.csv")
magasin = pd.read_csv(f"{DATA}/Dim_Magasin.csv")
client  = pd.read_csv(f"{DATA}/Dim_Client.csv")
dates   = pd.read_csv(f"{DATA}/Dim_Date.csv")
faits.head()

In [ ]:
faits.info()

## 2. Enrichir la table de faits (jointures)
On rapatrie les libellés des dimensions autour de la table de faits (comme un `merge`/jointure SQL).

In [ ]:
df = (faits
      .merge(produit[["produit_id","produit","categorie"]], on="produit_id", how="left")
      .merge(magasin[["magasin_id","ville","type"]], on="magasin_id", how="left")
      .merge(client[["client_id","segment"]], on="client_id", how="left")
      .merge(dates[["date_id","date","annee","mois","nom_mois","est_weekend"]], on="date_id", how="left"))
df["date"] = pd.to_datetime(df["date"])
df.shape

## 3. Qualité : valeurs manquantes et statistiques descriptives

In [ ]:
print("Valeurs manquantes par colonne :")
print(df.isna().sum())

In [ ]:
df[["quantite","prix_unitaire","remise","montant","marge"]].describe()

## 4. Distributions
Comment se répartissent le montant des ventes et les quantités ?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,4))
df["montant"].plot(kind="hist", bins=40, ax=ax[0], title="Distribution du montant (€)")
df["quantite"].plot(kind="hist", bins=20, ax=ax[1], title="Distribution des quantités")
plt.tight_layout(); plt.show()

## 5. Top catégories et magasins (agrégations)

In [ ]:
ca_categorie = df.groupby("categorie")["montant"].sum().sort_values(ascending=False)
ca_ville = df.groupby("ville")["montant"].sum().sort_values(ascending=False)
print("CA par catégorie :\n", ca_categorie.round(0), "\n")
print("CA par ville :\n", ca_ville.round(0))

In [ ]:
ca_categorie.plot(kind="bar", figsize=(9,4), title="Chiffre d'affaires par catégorie (€)")
plt.tight_layout(); plt.show()

## 6. Corrélations entre variables numériques

In [ ]:
num = df[["quantite","prix_unitaire","remise","montant","marge"]]
corr = num.corr()
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center")
plt.colorbar(im); plt.title("Matrice de corrélation"); plt.tight_layout(); plt.show()

## 🎯 À toi de jouer
Calcule le **chiffre d'affaires par segment client** (`segment`) et trie-le. Quel segment pèse le plus ?

In [ ]:
# Écris ta réponse ici :


<details><summary>💡 Corrigé</summary>

```python
df.groupby('segment')['montant'].sum().sort_values(ascending=False)
```
</details>

In [ ]:
df.groupby("segment")["montant"].sum().sort_values(ascending=False).round(0)

## ✅ À retenir
- On **enrichit** la table de faits par `merge` avant d'explorer.
- `info()` + `isna().sum()` + `describe()` = le triptyque de départ de toute EDA.
- `groupby` répond aux questions métier (« CA par… »).
- La matrice de corrélation repère les liens — sans confondre corrélation et causalité.